# 🏠 House Price Prediction — End-to-End Machine Learning Project

> **Kaggle Competition**: House Prices: Advanced Regression Techniques  
> **Objective**: Predict residential house sale prices using advanced regression techniques  
> **Dataset**: Ames, Iowa housing data — 79 features, 1460 training samples

---

## 📋 Table of Contents
1. [Setup & Imports](#1-setup)
2. [Data Loading & Inspection](#2-loading)
3. [Exploratory Data Analysis (EDA)](#3-eda)
4. [Data Cleaning](#4-cleaning)
5. [Feature Engineering](#5-features)
6. [Model Building](#6-models)
7. [Model Evaluation](#7-evaluation)
8. [Hyperparameter Tuning](#8-tuning)
9. [Feature Importance](#9-importance)
10. [Prediction & Submission](#10-prediction)

---
## 1. Setup & Imports <a id='1-setup'></a>

In [ ]:
# ─── Standard Library ────────────────────────────────────────────────────────
import os
import sys
import random
import warnings
import pickle
from pathlib import Path

warnings.filterwarnings('ignore')

# ─── Add project root to path ────────────────────────────────────────────────
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

# ─── Data Science ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import norm, skew

# ─── Visualization ───────────────────────────────────────────────────────────
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# ─── Machine Learning ────────────────────────────────────────────────────────
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import (
    train_test_split, cross_val_score, KFold,
    GridSearchCV, RandomizedSearchCV
)
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import RobustScaler
import joblib

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
    print('✅ XGBoost available')
except ImportError:
    HAS_XGB = False
    print('⚠️  XGBoost not installed')

try:
    from lightgbm import LGBMRegressor
    HAS_LGB = True
    print('✅ LightGBM available')
except ImportError:
    HAS_LGB = False
    print('⚠️  LightGBM not installed')

# ─── Project Modules ─────────────────────────────────────────────────────────
from src.utils import set_seed, save_figure, FIGURES_DIR
from src.preprocessing import HousePricePreprocessor
from src.feature_engineering import FeatureEngineer

# ─── Configuration ───────────────────────────────────────────────────────────
SEED = 42
set_seed(SEED)

# ─── Plot Style ──────────────────────────────────────────────────────────────
DARK_BG  = '#0f0f1a'
ACCENT   = '#7c5cbf'
ACCENT2  = '#e96c7d'
PALETTE  = ['#7c5cbf', '#e96c7d', '#5bc0eb', '#ffa500', '#4caf50',
            '#f06292', '#ab47bc', '#26c6da']
TEXT_CLR = '#e0e0e0'

plt.rcParams.update({
    'figure.facecolor': DARK_BG,
    'axes.facecolor':   '#1a1a2e',
    'axes.labelcolor':  TEXT_CLR,
    'axes.edgecolor':   '#3a3a5c',
    'text.color':       TEXT_CLR,
    'xtick.color':      TEXT_CLR,
    'ytick.color':      TEXT_CLR,
    'grid.color':       '#2a2a4a',
    'grid.alpha':       0.5,
    'legend.facecolor': '#1a1a2e',
    'legend.edgecolor': '#3a3a5c',
    'font.size':        11,
    'figure.dpi':       100,
})

print(f'\n🔧 Setup complete | Seed: {SEED} | Figures: {FIGURES_DIR}')

---
## 2. Data Loading & Inspection <a id='2-loading'></a>

In [ ]:
# ─── Load datasets ────────────────────────────────────────────────────────────
DATA_DIR = PROJECT_ROOT / 'data'

train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df  = pd.read_csv(DATA_DIR / 'test.csv')

print(f'🏠 Training set:  {train_df.shape[0]:,} rows × {train_df.shape[1]} columns')
print(f'🏠 Test set:      {test_df.shape[0]:,} rows × {test_df.shape[1]} columns')
print(f'\nTarget column: SalePrice')
print(f'  Min:  ${train_df["SalePrice"].min():>10,.0f}')
print(f'  Max:  ${train_df["SalePrice"].max():>10,.0f}')
print(f'  Mean: ${train_df["SalePrice"].mean():>10,.0f}')
print(f'  Std:  ${train_df["SalePrice"].std():>10,.0f}')

In [ ]:
# ─── First look ───────────────────────────────────────────────────────────────
print('=== First 5 rows ===')
display(train_df.head())

In [ ]:
# ─── Feature types ────────────────────────────────────────────────────────────
num_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = train_df.select_dtypes(include=['object']).columns.tolist()

print(f'Numerical features: {len(num_cols)}')
print(f'Categorical features: {len(cat_cols)}')
print(f'\nNumerical columns:\n{num_cols}')
print(f'\nCategorical columns:\n{cat_cols}')

In [ ]:
# ─── Summary statistics ───────────────────────────────────────────────────────
display(train_df.describe().T.style.background_gradient(cmap='Purples'))

In [ ]:
# ─── Data types overview ──────────────────────────────────────────────────────
dtype_summary = pd.DataFrame({
    'dtype': train_df.dtypes,
    'non_null': train_df.count(),
    'null_count': train_df.isnull().sum(),
    'null_pct': (train_df.isnull().mean() * 100).round(2),
    'unique': train_df.nunique(),
})
display(dtype_summary[dtype_summary['null_count'] > 0].sort_values('null_pct', ascending=False))

---
## 3. Exploratory Data Analysis (EDA) <a id='3-eda'></a>

In [ ]:
# ─── 3.1 SalePrice Distribution ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('SalePrice Distribution', fontsize=16, fontweight='bold', y=1.02)

# Original
axes[0].hist(train_df['SalePrice'], bins=60, color=ACCENT, edgecolor='none', alpha=0.85)
axes[0].set_title('Original SalePrice', fontsize=13)
axes[0].set_xlabel('Sale Price ($)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(train_df['SalePrice'].mean(), color=ACCENT2, lw=2.5, linestyle='--',
                label=f'Mean: ${train_df["SalePrice"].mean():,.0f}')
axes[0].axvline(train_df['SalePrice'].median(), color='#5bc0eb', lw=2.5, linestyle='-.',
                label=f'Median: ${train_df["SalePrice"].median():,.0f}')
axes[0].legend()

# Log-transformed
log_price = np.log1p(train_df['SalePrice'])
axes[1].hist(log_price, bins=60, color=ACCENT2, edgecolor='none', alpha=0.85)
axes[1].set_title('Log1p(SalePrice) — Normalized', fontsize=13)
axes[1].set_xlabel('log1p(Sale Price)')
axes[1].set_ylabel('Frequency')

# Fit normal curve
mu, std = norm.fit(log_price)
x = np.linspace(log_price.min(), log_price.max(), 200)
axes[1].plot(x, norm.pdf(x, mu, std) * len(log_price) * (log_price.max()-log_price.min())/60,
             color='white', lw=2, alpha=0.7, label='Normal fit')
axes[1].legend()

plt.tight_layout()
save_figure(fig, 'saleprice_distribution')
plt.show()
print(f'\nSkewness (original): {train_df["SalePrice"].skew():.4f}')
print(f'Skewness (log):      {log_price.skew():.4f}')

In [ ]:
# ─── 3.2 Missing Value Heatmap ───────────────────────────────────────────────
missing = train_df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(train_df) * 100).round(2)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle('Missing Value Analysis', fontsize=16, fontweight='bold')

# Bar chart of missing counts
colors = plt.cm.plasma(np.linspace(0.2, 0.9, len(missing)))
axes[0].barh(missing.index, missing.values, color=colors, edgecolor='none')
axes[0].set_xlabel('Missing Count', fontsize=12)
axes[0].set_title('Columns with Missing Values', fontsize=13)
for i, (val, pct) in enumerate(zip(missing.values, missing_pct.values)):
    axes[0].text(val + 5, i, f'{val} ({pct}%)', va='center', fontsize=8.5)

# Heatmap of first 50 rows missing pattern
miss_cols = missing.index.tolist()[:20]
miss_data = train_df[miss_cols].isnull().astype(int).head(80)
sns.heatmap(
    miss_data.T, ax=axes[1], cmap='RdPu', cbar=False,
    xticklabels=False, yticklabels=True
)
axes[1].set_title('Missing Pattern (first 80 rows)', fontsize=13)
axes[1].set_xlabel('Sample Index')

plt.tight_layout()
save_figure(fig, 'missing_values')
plt.show()

In [ ]:
# ─── 3.3 Correlation Heatmap ─────────────────────────────────────────────────
corr_features = [
    'SalePrice', 'OverallQual', 'GrLivArea', 'GarageCars', 'GarageArea',
    'TotalBsmtSF', '1stFlrSF', 'FullBath', 'TotRmsAbvGrd', 'YearBuilt',
    'YearRemodAdd', 'GarageYrBlt', 'MasVnrArea', 'Fireplaces',
    'LotFrontage', 'WoodDeckSF', 'OpenPorchSF', 'LotArea',
    'BedroomAbvGr', 'KitchenAbvGr',
]
corr_df = train_df[[c for c in corr_features if c in train_df.columns]].corr()

fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(corr_df, dtype=bool))

sns.heatmap(
    corr_df, ax=ax, mask=mask,
    cmap='RdPu', center=0,
    square=True, linewidths=0.5, linecolor='#0f0f1a',
    annot=True, fmt='.2f', annot_kws={'size': 8, 'color': 'white'},
    vmin=-1, vmax=1,
    cbar_kws={'shrink': 0.8, 'label': 'Pearson Correlation'},
)
ax.set_title('Feature Correlation Heatmap', fontsize=16, fontweight='bold', pad=20)
ax.tick_params(axis='x', rotation=45, labelsize=9)
ax.tick_params(axis='y', rotation=0, labelsize=9)

plt.tight_layout()
save_figure(fig, 'correlation_heatmap')
plt.show()

In [ ]:
# ─── 3.4 Top Correlated Features with SalePrice ──────────────────────────────
top_corr = corr_df['SalePrice'].abs().sort_values(ascending=False).drop('SalePrice').head(15)

fig, ax = plt.subplots(figsize=(12, 7))
colors = [ACCENT if v > 0 else ACCENT2 for v in corr_df['SalePrice'][top_corr.index]]
ax.barh(top_corr.index, corr_df['SalePrice'][top_corr.index], color=colors, edgecolor='none', height=0.7)
ax.set_xlabel('Pearson Correlation with SalePrice', fontsize=12)
ax.set_title('Top 15 Features Correlated with SalePrice', fontsize=14, fontweight='bold')
ax.axvline(0, color='white', lw=1, alpha=0.3)
ax.grid(True, axis='x', alpha=0.2)
for i, v in enumerate(corr_df['SalePrice'][top_corr.index]):
    ax.text(v + (0.005 if v > 0 else -0.005), i,
            f'{v:.3f}', va='center', ha='left' if v > 0 else 'right', fontsize=9)
plt.tight_layout()
save_figure(fig, 'top_correlations')
plt.show()

In [ ]:
# ─── 3.5 Key Features vs SalePrice Scatterplots ──────────────────────────────
key_features = ['GrLivArea', 'OverallQual', 'TotalBsmtSF', 'YearBuilt', 'GarageArea', 'LotArea']

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Key Features vs. SalePrice', fontsize=16, fontweight='bold')
axes = axes.flatten()

for i, feat in enumerate(key_features):
    if feat in train_df.columns:
        sc = axes[i].scatter(
            train_df[feat], train_df['SalePrice'],
            alpha=0.5, s=15, c=train_df['OverallQual'],
            cmap='plasma', edgecolors='none'
        )
        axes[i].set_xlabel(feat, fontsize=11)
        axes[i].set_ylabel('SalePrice ($)', fontsize=11)
        axes[i].set_title(f'{feat} vs. SalePrice', fontsize=12, fontweight='bold')
        axes[i].grid(True, alpha=0.15)

plt.colorbar(sc, ax=axes[-1], label='OverallQual')
plt.tight_layout()
save_figure(fig, 'features_vs_saleprice')
plt.show()

In [ ]:
# ─── 3.6 OverallQual vs SalePrice (key driver) ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Overall Quality — Primary Value Driver', fontsize=15, fontweight='bold')

# Boxplot
qual_groups = [train_df[train_df['OverallQual'] == q]['SalePrice'].values
               for q in sorted(train_df['OverallQual'].unique())]
bp = axes[0].boxplot(qual_groups, labels=sorted(train_df['OverallQual'].unique()),
                     patch_artist=True, medianprops=dict(color='white', lw=2))
for patch, color in zip(bp['boxes'], plt.cm.plasma(np.linspace(0.1, 0.9, len(qual_groups)))):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
axes[0].set_xlabel('Overall Quality', fontsize=12)
axes[0].set_ylabel('Sale Price ($)', fontsize=12)
axes[0].set_title('Price Distribution by Quality', fontsize=13)
axes[0].grid(True, axis='y', alpha=0.2)

# Mean price by quality
means = train_df.groupby('OverallQual')['SalePrice'].mean()
colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(means)))
axes[1].bar(means.index, means.values, color=colors, edgecolor='none', width=0.7)
axes[1].set_xlabel('Overall Quality', fontsize=12)
axes[1].set_ylabel('Mean Sale Price ($)', fontsize=12)
axes[1].set_title('Mean Price by Quality Score', fontsize=13)
axes[1].grid(True, axis='y', alpha=0.2)
for i, (q, v) in enumerate(means.items()):
    axes[1].text(q, v + 2000, f'${v/1000:.0f}K', ha='center', fontsize=8.5)

plt.tight_layout()
save_figure(fig, 'quality_vs_price')
plt.show()

In [ ]:
# ─── 3.7 Neighborhood Analysis ───────────────────────────────────────────────
nbhd_stats = train_df.groupby('Neighborhood')['SalePrice'].agg(['median', 'mean', 'count'])
nbhd_stats = nbhd_stats.sort_values('median', ascending=True)

fig, ax = plt.subplots(figsize=(14, 10))
colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(nbhd_stats)))

bars = ax.barh(nbhd_stats.index, nbhd_stats['median'], color=colors, edgecolor='none', height=0.7)
ax.set_xlabel('Median Sale Price ($)', fontsize=12)
ax.set_title('Median House Price by Neighborhood', fontsize=14, fontweight='bold', pad=15)
ax.grid(True, axis='x', alpha=0.2)

for i, (idx, row) in enumerate(nbhd_stats.iterrows()):
    ax.text(row['median'] + 2000, i, f'${row["median"]/1000:.0f}K (n={row["count"]:.0f})',
            va='center', fontsize=8.5)

plt.tight_layout()
save_figure(fig, 'neighborhood_prices')
plt.show()

In [ ]:
# ─── 3.8 Outlier Detection ───────────────────────────────────────────────────
outlier_features = ['GrLivArea', 'LotArea', 'TotalBsmtSF', '1stFlrSF']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Outlier Detection — Key Numeric Features', fontsize=15, fontweight='bold')
axes = axes.flatten()

for i, feat in enumerate(outlier_features):
    if feat in train_df.columns:
        # Scatter with outlier coloring
        Q1 = train_df[feat].quantile(0.25)
        Q3 = train_df[feat].quantile(0.75)
        IQR = Q3 - Q1
        is_outlier = (train_df[feat] < Q1 - 1.5*IQR) | (train_df[feat] > Q3 + 1.5*IQR)

        axes[i].scatter(train_df.loc[~is_outlier, feat], train_df.loc[~is_outlier, 'SalePrice'],
                        alpha=0.45, s=12, color=ACCENT, label='Normal', edgecolors='none')
        axes[i].scatter(train_df.loc[is_outlier, feat], train_df.loc[is_outlier, 'SalePrice'],
                        alpha=0.8, s=40, color=ACCENT2, label=f'Outliers ({is_outlier.sum()})',
                        edgecolors='white', linewidths=0.5, marker='D')
        axes[i].set_xlabel(feat, fontsize=11)
        axes[i].set_ylabel('SalePrice ($)', fontsize=11)
        axes[i].set_title(f'{feat} vs. SalePrice', fontsize=12, fontweight='bold')
        axes[i].legend(fontsize=9)
        axes[i].grid(True, alpha=0.15)

plt.tight_layout()
save_figure(fig, 'outlier_detection')
plt.show()

In [ ]:
# ─── 3.9 Numerical Feature Distributions ────────────────────────────────────
key_num_features = ['GrLivArea', 'LotArea', 'TotalBsmtSF', '1stFlrSF',
                    'GarageArea', 'MasVnrArea', 'LotFrontage', 'OpenPorchSF']
key_num_features = [f for f in key_num_features if f in train_df.columns]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Numerical Feature Distributions', fontsize=15, fontweight='bold')
axes = axes.flatten()

for i, feat in enumerate(key_num_features):
    skewness = train_df[feat].skew()
    axes[i].hist(train_df[feat].dropna(), bins=40, color=PALETTE[i % len(PALETTE)],
                 edgecolor='none', alpha=0.85)
    axes[i].set_title(f'{feat}\nSkew: {skewness:.2f}', fontsize=10, fontweight='bold')
    axes[i].set_ylabel('Frequency', fontsize=9)
    axes[i].grid(True, alpha=0.2)

plt.tight_layout()
save_figure(fig, 'numerical_distributions')
plt.show()

In [ ]:
# ─── 3.10 Pairplot (Top 5 features) ─────────────────────────────────────────
pairplot_features = ['SalePrice', 'GrLivArea', 'OverallQual', 'TotalBsmtSF', 'GarageArea', 'YearBuilt']
pairplot_features = [f for f in pairplot_features if f in train_df.columns]

pp_df = train_df[pairplot_features].dropna().sample(400, random_state=SEED)

with plt.style.context('dark_background'):
    pp = sns.pairplot(
        pp_df, diag_kind='kde',
        plot_kws={'alpha': 0.5, 's': 10, 'color': ACCENT},
        diag_kws={'color': ACCENT2, 'fill': True},
    )
    pp.fig.suptitle('Pairplot — Key Features', y=1.02, fontsize=14, fontweight='bold')

save_figure(pp.fig, 'pairplot')
plt.show()

---
## 4. Data Cleaning <a id='4-cleaning'></a>

In [ ]:
# ─── Separate target ─────────────────────────────────────────────────────────
y_raw = train_df['SalePrice'].copy()
X_raw = train_df.drop(columns=['SalePrice']).copy()

# ─── Apply preprocessor ──────────────────────────────────────────────────────
preprocessor = HousePricePreprocessor(iqr_multiplier=1.5)
X_clean = preprocessor.fit_transform(X_raw)

print(f'Before cleaning: {X_raw.shape}')
print(f'After cleaning:  {X_clean.shape}')
print(f'\nRemaining nulls: {X_clean.isnull().sum().sum()}')

In [ ]:
# ─── Verify no nulls remain ───────────────────────────────────────────────────
null_after = X_clean.isnull().sum()
print('Columns with nulls after cleaning:')
print(null_after[null_after > 0] if null_after.sum() > 0 else '✅ None — all nulls handled!')

---
## 5. Feature Engineering <a id='5-features'></a>

In [ ]:
# ─── Log-transform target ─────────────────────────────────────────────────────
y = np.log1p(y_raw)
print(f'Target transformation: log1p(SalePrice)')
print(f'  Original skewness: {y_raw.skew():.4f}')
print(f'  Log skewness:      {y.skew():.4f}')

# ─── Train/val split before feature engineering ───────────────────────────────
X_tr_raw, X_val_raw, y_tr, y_val = train_test_split(
    X_clean, y, test_size=0.2, random_state=SEED
)
print(f'\nTrain: {X_tr_raw.shape}, Val: {X_val_raw.shape}')

In [ ]:
# ─── Apply FeatureEngineer ────────────────────────────────────────────────────
feat_eng = FeatureEngineer(skew_threshold=0.75, corr_threshold=0.95, scale=True)
X_train = feat_eng.fit_transform(X_tr_raw)
X_val   = feat_eng.transform(X_val_raw)

print(f'\nAfter feature engineering:')
print(f'  Train: {X_train.shape}')
print(f'  Val:   {X_val.shape}')
print(f'\nNew features created:')
new_features = ['TotalSF', 'TotalBaths', 'HouseAge', 'RemodAge', 'IsRemodeled',
                'HasPool', 'HasFireplace', 'HasGarage', 'QualArea', 'QualTotalSF',
                'TotalPorchSF', 'Has2ndFloor', 'HasBsmt', 'HasMasVnr', 'HasWoodDeck']
present = [f for f in new_features if f in X_train.columns]
print(f'  {present}')

In [ ]:
# ─── Skewness correction visualization ──────────────────────────────────────
print(f'Features corrected for skewness: {len(feat_eng.skewed_cols_)}')
print(f'Low-variance features dropped:  {len(feat_eng.dropped_low_var_)}')
print(f'High-correlation features dropped: {len(feat_eng.dropped_high_corr_)}')

---
## 6. Model Building <a id='6-models'></a>

In [ ]:
# ─── Define all models ────────────────────────────────────────────────────────
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression':  Ridge(alpha=1.0, random_state=SEED),
    'Lasso Regression':  Lasso(alpha=0.001, max_iter=10000, random_state=SEED),
    'Decision Tree':     DecisionTreeRegressor(max_depth=10, random_state=SEED),
    'Random Forest':     RandomForestRegressor(
        n_estimators=200, max_depth=15, min_samples_leaf=2,
        n_jobs=-1, random_state=SEED,
    ),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=5,
        subsample=0.8, random_state=SEED,
    ),
}

if HAS_XGB:
    models['XGBoost'] = XGBRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0,
        n_jobs=-1, random_state=SEED, verbosity=0,
    )

if HAS_LGB:
    models['LightGBM'] = LGBMRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=6,
        num_leaves=50, subsample=0.8, colsample_bytree=0.8,
        n_jobs=-1, random_state=SEED, verbosity=-1,
    )

print(f'Models to train: {list(models.keys())}')

In [ ]:
# ─── Train all models and collect metrics ─────────────────────────────────────
results = []
trained_models = {}
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

for name, model in models.items():
    print(f'\nTraining: {name}...')
    model.fit(X_train, y_tr)
    trained_models[name] = model

    # Predictions
    y_pred     = np.expm1(model.predict(X_val))
    y_true_exp = np.expm1(y_val.values)

    r2   = r2_score(y_true_exp, y_pred)
    mae  = mean_absolute_error(y_true_exp, y_pred)
    mse  = mean_squared_error(y_true_exp, y_pred)
    rmse = np.sqrt(mse)

    # Cross-validation
    cv_r2   = cross_val_score(model, X_train, y_tr, cv=kf, scoring='r2', n_jobs=-1).mean()
    cv_rmse = -cross_val_score(model, X_train, y_tr, cv=kf,
                               scoring='neg_root_mean_squared_error', n_jobs=-1).mean()

    results.append({
        'model': name, 'R2': r2, 'MAE': mae, 'MSE': mse,
        'RMSE': rmse, 'CV_R2': cv_r2, 'CV_RMSE': cv_rmse,
    })
    print(f'  R²={r2:.4f}  MAE=${mae:,.0f}  RMSE=${rmse:,.0f}  CV_R²={cv_r2:.4f}')

results_df = pd.DataFrame(results).sort_values('R2', ascending=False).reset_index(drop=True)
print('\n✅ All models trained!')

---
## 7. Model Evaluation <a id='7-evaluation'></a>

In [ ]:
# ─── Results Table ───────────────────────────────────────────────────────────
display(results_df.style
    .background_gradient(subset=['R2', 'CV_R2'], cmap='Purples')
    .background_gradient(subset=['RMSE', 'MAE'], cmap='RdPu_r')
    .format({'R2': '{:.4f}', 'MAE': '${:,.0f}', 'MSE': '${:,.0f}',
             'RMSE': '${:,.0f}', 'CV_R2': '{:.4f}', 'CV_RMSE': '{:.4f}'})
)

In [ ]:
# ─── Model Comparison Visualization ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 7))
fig.suptitle('Model Performance Comparison', fontsize=16, fontweight='bold')

df_sorted = results_df.sort_values('R2', ascending=True)
colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(df_sorted)))

# R²
axes[0].barh(df_sorted['model'], df_sorted['R2'], color=colors, edgecolor='none', height=0.65)
axes[0].set_xlabel('R² Score', fontsize=11)
axes[0].set_title('R² Score', fontsize=13, fontweight='bold')
for i, v in enumerate(df_sorted['R2']):
    axes[0].text(v + 0.003, i, f'{v:.4f}', va='center', fontsize=9)
axes[0].grid(True, axis='x', alpha=0.2)

# RMSE
axes[1].barh(df_sorted['model'], df_sorted['RMSE'], color=colors, edgecolor='none', height=0.65)
axes[1].set_xlabel('RMSE ($)', fontsize=11)
axes[1].set_title('RMSE (lower is better)', fontsize=13, fontweight='bold')
for i, v in enumerate(df_sorted['RMSE']):
    axes[1].text(v + 200, i, f'${v:,.0f}', va='center', fontsize=9)
axes[1].grid(True, axis='x', alpha=0.2)

# CV R²
axes[2].barh(df_sorted['model'], df_sorted['CV_R2'], color=colors, edgecolor='none', height=0.65)
axes[2].set_xlabel('CV R² Score', fontsize=11)
axes[2].set_title('Cross-Validation R²', fontsize=13, fontweight='bold')
for i, v in enumerate(df_sorted['CV_R2']):
    axes[2].text(v + 0.003, i, f'{v:.4f}', va='center', fontsize=9)
axes[2].grid(True, axis='x', alpha=0.2)

plt.tight_layout()
save_figure(fig, 'model_comparison_notebook')
plt.show()

In [ ]:
# ─── Actual vs Predicted (Best Model) ────────────────────────────────────────
best_name  = results_df.iloc[0]['model']
best_model = trained_models[best_name]
print(f'Best model: {best_name}')

y_pred_best  = np.expm1(best_model.predict(X_val))
y_true_orig  = np.expm1(y_val.values)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle(f'Best Model: {best_name}', fontsize=15, fontweight='bold')

# Actual vs Predicted
axes[0].scatter(y_true_orig, y_pred_best, alpha=0.5, s=15, color=ACCENT, edgecolors='none')
lims = [min(y_true_orig.min(), y_pred_best.min()), max(y_true_orig.max(), y_pred_best.max())]
axes[0].plot(lims, lims, 'w--', lw=2, alpha=0.6, label='Perfect')
axes[0].set_xlabel('Actual Sale Price ($)', fontsize=12)
axes[0].set_ylabel('Predicted Sale Price ($)', fontsize=12)
axes[0].set_title('Actual vs. Predicted', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.15)

# Residuals
residuals = y_true_orig - y_pred_best
axes[1].scatter(y_pred_best, residuals, alpha=0.5, s=15, color=ACCENT2, edgecolors='none')
axes[1].axhline(0, color='white', lw=2, linestyle='--', alpha=0.6)
axes[1].set_xlabel('Predicted Sale Price ($)', fontsize=12)
axes[1].set_ylabel('Residuals ($)', fontsize=12)
axes[1].set_title('Residual Plot', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.15)

plt.tight_layout()
save_figure(fig, 'actual_vs_predicted_notebook')
plt.show()

---
## 8. Hyperparameter Tuning <a id='8-tuning'></a>

In [ ]:
# ─── Tune the best model ─────────────────────────────────────────────────────
print(f'Tuning: {best_name}')

if best_name == 'XGBoost' and HAS_XGB:
    param_dist = {
        'n_estimators': [200, 300, 400],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [4, 5, 6, 7],
        'subsample': [0.7, 0.8, 0.9],
        'colsample_bytree': [0.7, 0.8, 0.9],
        'reg_alpha': [0, 0.1, 0.5],
        'reg_lambda': [0.5, 1.0, 2.0],
    }
    param_grid = {
        'n_estimators': [300, 400],
        'learning_rate': [0.05, 0.1],
        'max_depth': [5, 6],
    }
elif best_name == 'Gradient Boosting':
    param_dist = {
        'n_estimators': [200, 300, 400],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [3, 4, 5, 6],
        'subsample': [0.7, 0.8, 0.9],
    }
    param_grid = {
        'n_estimators': [300, 400],
        'learning_rate': [0.05, 0.1],
        'max_depth': [4, 5],
    }
else:
    param_dist = {
        'n_estimators': [100, 200, 300],
        'max_depth': [None, 10, 15, 20],
        'min_samples_leaf': [1, 2, 4],
    }
    param_grid = {
        'n_estimators': [200, 300],
        'max_depth': [15, 20],
    }

# RandomizedSearchCV — broad exploration
rnd_search = RandomizedSearchCV(
    best_model, param_dist, n_iter=20,
    scoring='neg_root_mean_squared_error',
    cv=kf, n_jobs=-1, random_state=SEED, verbose=1
)
rnd_search.fit(X_train, y_tr)
print(f'\nRandomizedSearchCV best CV RMSE: {-rnd_search.best_score_:.6f}')
print(f'Best params (random): {rnd_search.best_params_}')

In [ ]:
# GridSearchCV — fine-tuning
grid_search = GridSearchCV(
    rnd_search.best_estimator_, param_grid,
    scoring='neg_root_mean_squared_error',
    cv=kf, n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_tr)
tuned_model = grid_search.best_estimator_

print(f'\nGridSearchCV best CV RMSE: {-grid_search.best_score_:.6f}')
print(f'Best params (grid): {grid_search.best_params_}')

# Evaluate tuned model
y_pred_tuned = np.expm1(tuned_model.predict(X_val))
tuned_r2   = r2_score(y_true_orig, y_pred_tuned)
tuned_rmse = np.sqrt(mean_squared_error(y_true_orig, y_pred_tuned))

print(f'\n✅ Tuned model — R²={tuned_r2:.4f}, RMSE=${tuned_rmse:,.0f}')

---
## 9. Feature Importance <a id='9-importance'></a>

In [ ]:
# ─── Feature Importance Chart ────────────────────────────────────────────────
if hasattr(tuned_model, 'feature_importances_'):
    feature_names = list(X_train.columns)
    importances = tuned_model.feature_importances_
    top_n = 30

    indices = np.argsort(importances)[::-1][:top_n]
    top_feats = [feature_names[i] for i in indices]
    top_imps  = importances[indices]

    fig, ax = plt.subplots(figsize=(13, 11))
    colors = plt.cm.plasma(np.linspace(0.2, 0.9, top_n))
    ax.barh(top_feats[::-1], top_imps[::-1], color=colors[::-1], edgecolor='none')
    ax.set_xlabel('Feature Importance', fontsize=12)
    ax.set_title(f'Top {top_n} Feature Importances — {best_name} (Tuned)',
                 fontsize=14, fontweight='bold', pad=15)
    ax.grid(True, axis='x', alpha=0.2)

    # Annotate
    for i, v in enumerate(top_imps[::-1]):
        ax.text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=8)

    plt.tight_layout()
    save_figure(fig, 'feature_importance_tuned_notebook')
    plt.show()

    print(f'\n🔑 Top 10 most important features:')
    for feat, imp in zip(top_feats[:10], top_imps[:10]):
        print(f'   {feat:<35}: {imp:.6f}')
else:
    print('⚠️ This model does not expose feature_importances_')

---
## 10. Prediction & Submission <a id='10-prediction'></a>

In [ ]:
# ─── Save trained pipeline ───────────────────────────────────────────────────
MODELS_DIR = PROJECT_ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)

pipeline = {
    'preprocessor':     preprocessor,
    'feature_engineer': feat_eng,
    'model':            tuned_model,
}

# Joblib
joblib.dump(pipeline, MODELS_DIR / 'house_price_pipeline.joblib')
# Pickle
with open(MODELS_DIR / 'house_price_model.pkl', 'wb') as f:
    pickle.dump(tuned_model, f)
# Joblib model only
joblib.dump(tuned_model, MODELS_DIR / 'house_price_model.joblib')

print('✅ Models saved:')
for f in MODELS_DIR.glob('*'):
    print(f'   {f.name} ({f.stat().st_size // 1024} KB)')

In [ ]:
# ─── Generate Kaggle submission ───────────────────────────────────────────────
test_path = DATA_DIR / 'test.csv'

if test_path.exists():
    test_raw = pd.read_csv(test_path)
    ids = test_raw['Id'].copy()

    # Full pipeline
    X_test_clean = preprocessor.transform(test_raw)
    X_test_fe    = feat_eng.transform(X_test_clean)
    test_pred    = np.expm1(tuned_model.predict(X_test_fe))
    test_pred    = np.clip(test_pred, 0, None)

    submission = pd.DataFrame({'Id': ids, 'SalePrice': test_pred})

    OUTPUT_DIR = PROJECT_ROOT / 'outputs'
    OUTPUT_DIR.mkdir(exist_ok=True)
    sub_path = OUTPUT_DIR / 'submission.csv'
    submission.to_csv(sub_path, index=False)

    print(f'✅ Submission saved → {sub_path}')
    print(f'\nPrediction Statistics:')
    print(submission['SalePrice'].describe().apply(lambda x: f'${x:,.2f}' if isinstance(x, float) else x))
    display(submission.head(10))
else:
    print(f'⚠️ test.csv not found at {test_path}')
    print('  Place the Kaggle test.csv in the data/ folder to generate a submission.')

In [ ]:
# ─── Final Summary ────────────────────────────────────────────────────────────
print('=' * 65)
print('  🏠 House Price Prediction — Project Summary')
print('=' * 65)
print(f'  Dataset:          Ames Housing (Kaggle)')
print(f'  Training samples: {len(X_train):,}')
print(f'  Features used:    {X_train.shape[1]}')
print(f'  Best model:       {best_name} (Tuned)')
print(f'  Validation R²:    {tuned_r2:.4f}')
print(f'  Validation RMSE:  ${tuned_rmse:,.0f}')
print('=' * 65)
print(f'  Plots saved to:   outputs/figures/')
print(f'  Model saved to:   models/')
print(f'  Submission:       outputs/submission.csv')
print('=' * 65)